# Tokenization & Materialization — Interactive Exploration

This notebook experiments with the tokenization pipeline and materialization
(token reconstruction) using the HLLSet DSL.

**Pipeline:** `Bytes → [Pattern Match] → [Normalize] → [N-grams] → HLLSet → Materialize`

In [ ]:
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-dsl" }
use hllset_dsl::*;
use hllset_dsl::pattern::Pattern;

println!("✅ hllset-dsl loaded")

---
## 1. Basic Tokenization

Default tokenizer extracts words (ASCII letters + digits).

In [ ]:
let tok = Tokenizer::new();
let tokens = tok.extract(b"The cat sat on the mat");
for (i, t) in tokens.iter().enumerate() {
    println!("token {}: {:?}", i + 1, std::str::from_utf8(t).unwrap());
}

In [ ]:
let tok = Tokenizer::new().lowercase();
let tokens = tok.tokenize(b"The CAT sat ON the MAT");
for t in &tokens {
    print!("{} ", std::str::from_utf8(t).unwrap());
}
println!()

In [ ]:
let tok = Tokenizer::new().lowercase();
let elem: LatticeElement = tok.apply(b"Hello World Lua DSL");
println!("key:          {}", elem.key());
println!("cardinality:  {:.1}", elem.cardinality());
println!("popcount:     {}", elem.popcount());

---
## 2. N-gram Tokenization

N-grams capture local context. Bigrams/trigrams join adjacent tokens with NUL (0x00).

In [ ]:
let tok = Tokenizer::new().lowercase().ngrams(1, 3);
let tokens = tok.tokenize(b"the cat sat on the mat");
for t in &tokens {
    let s = String::from_utf8_lossy(t).replace("\0", "|");
    println!("  {}", s);
}

In [ ]:
let tok = Tokenizer::new().lowercase().ngrams(1, 3);
let elem: LatticeElement = tok.apply(b"the cat sat");
println!("N-grams 1..3: {} bits, cardinality {:.1}",
         elem.popcount(), elem.cardinality());

---
## 3. Boundary Padding

START/END markers enable context at sequence boundaries.

In [ ]:
let tok = Tokenizer::new().lowercase()
    .pad(b"<S>", b"</S>").ngrams(1, 3);
let tokens = tok.tokenize(b"hello world");
for t in &tokens {
    let s = String::from_utf8_lossy(t).replace("\0", " → ");
    println!("  {}", s);
}

---
## 4. Custom SNOBOL Patterns

Composable matchers for arbitrary byte sequences.

In [ ]:
let digit = Pattern::span(b"0123456789");
let tok = Tokenizer::new().pattern(digit);
let tokens = tok.extract(b"x=42, y=137, z=999");
for t in &tokens { print!("{} ", std::str::from_utf8(t).unwrap()); }
println!()

In [ ]:
let alpha = Pattern::span(b"abcdefghijklmnopqrstuvwxyz");
let digit = Pattern::span(b"0123456789");
let tok = Tokenizer::new().pattern(alpha.alt(digit)).lowercase();
let elem: LatticeElement = tok.apply(b"word1 1234 word2 5678");
println!("popcount: {}", elem.popcount());

---
## 5. Materialization — InLUT Strategy

Reconstructs tokens from HLLSet using a reverse index (TokenLUT).

In [ ]:
let tok = Tokenizer::new().lowercase();
let hllset: HLLSet = tok.apply(b"alpha beta gamma").into_hllset();
let tokens = tok.tokenize(b"alpha beta gamma");
let lut = TokenLUT::from_tokens(&tokens);
let result = materialize_inlut(&hllset, &lut);
println!("Strategy:   {}", result.strategy);
println!("Confidence: {:.2}%", result.confidence * 100.0);
println!("Tokens:     {:?}", result.flat_strings());

In [ ]:
let input = "lattice algebra bss morphism tokenization materialization";
let tok = Tokenizer::new().lowercase();
let hllset: HLLSet = tok.apply(input.as_bytes()).into_hllset();
let all_tokens = tok.tokenize(input.as_bytes());
let lut = TokenLUT::from_tokens(&all_tokens);
let result = materialize_inlut(&hllset, &lut);
let flat = result.flat_strings();
let missing: Vec<&str> = input.split_whitespace()
    .filter(|w| !flat.contains(&w.to_lowercase()))
    .collect();
if missing.is_empty() {
    println!("✅ Full roundtrip! All {} words recovered.", input.split_whitespace().count());
} else {
    println!("⚠ Missing: {:?}", missing);
}

In [ ]:
let text = "the quick brown fox jumps over the lazy dog";
let tok = Tokenizer::new().lowercase().ngrams(1, 2);
let hllset: HLLSet = tok.apply(text.as_bytes()).into_hllset();
let all_tokens = tok.tokenize(text.as_bytes());
let full_lut = TokenLUT::from_tokens(&all_tokens);
let r1 = materialize_inlut(&hllset, &full_lut);
println!("Full LUT:    conf {:.2}%, {} tokens", r1.confidence*100.0, r1.flat_strings().len());
let unigrams = tok.extract(text.as_bytes());
let partial_lut = TokenLUT::from_tokens(&unigrams);
let r2 = materialize_inlut(&hllset, &partial_lut);
println!("Partial LUT: conf {:.2}%, {} tokens", r2.confidence*100.0, r2.flat_strings().len());

---
## 6. N-gram Cross Validation

Validates bigrams by checking constituent unigrams are attested.

In [ ]:
let tok = Tokenizer::new().lowercase().ngrams(1, 2);
let hllset: HLLSet = tok.apply(b"the cat sat").into_hllset();
let tokens = tok.tokenize(b"the cat sat");
let lut = TokenLUT::from_tokens(&tokens);
let result = materialize_ngram_cross_validate(&hllset, &lut);
println!("Strategy:   {}", result.strategy);
println!("Confidence: {:.2}%", result.confidence * 100.0);
for t in &result.candidates[0] {
    let s = String::from_utf8_lossy(t).replace("\0", "|");
    println!("  {}", s);
}

---
## 7. De Bruijn Graph Reconstruction

Reconstructs token ORDER by finding a path through the n-gram graph.

In [ ]:
let tok = Tokenizer::new().lowercase()
    .pad(b"<S>", b"</S>").ngrams(2, 2);
let hllset: HLLSet = tok.apply(b"the cat sat on the mat").into_hllset();
let tokens = tok.tokenize(b"the cat sat on the mat");
let lut = TokenLUT::from_tokens(&tokens);
let result = materialize_debruijn(&hllset, &lut, b"<S>", b"</S>");
println!("Strategy:   {}", result.strategy);
println!("Confidence: {:.1}%", result.confidence * 100.0);
for seq in &result.candidates {
    let path: Vec<String> = seq.iter()
        .map(|t| String::from_utf8_lossy(t).to_string()).collect();
    println!("Path: {}", path.join(" → "));
}

---
## 8. Lua DSL Pipeline

Same operations via Lua through `DslRuntime`.

In [ ]:
let mut rt = DslRuntime::new().unwrap();
rt.exec(r#"
    local elem = hllset.tokenize("hello world lua dsl")
    print("key: " .. elem:key())
    print("card: " .. #elem)
    local result = hllset.materialize(elem, {"hello","world","lua","dsl"})
    print("confidence: " .. result.confidence)
    for i = 1, #result.tokens do
        print("  " .. result.tokens[i])
    end
"#).unwrap();

In [ ]:
let mut rt = DslRuntime::new().unwrap();
rt.exec(r#"
    local tok = hllset.tokenizer()
    tok:lowercase()
    tok:pad("<S>", "</S>")
    tok:ngrams(1, 2)
    local elem = tok:apply("the cat sat")
    print("cardinality: " .. #elem)
    local tokens = tok:tokenize("the cat sat")
    print("tokens extracted: " .. #tokens)
"#).unwrap();

In [ ]:
let mut rt = DslRuntime::new().unwrap();
rt.exec(r#"
    local a = hllset.tokenize("the cat sat on the mat")
    local b = hllset.tokenize("the dog ran in the park")
    local shared = a * b
    local combined = a + b
    print("shared: " .. #shared .. "  combined: " .. #combined)
    print("BSS inclusion: " .. a:bss_inclusion(b))
    print("Jaccard: " .. a:jaccard(b))
"#).unwrap();

---
## 9. Document Comparison via HLLSet

HLLSets enable O(1) set comparisons for document similarity.

In [ ]:
{
let tok = Tokenizer::new().lowercase().ngrams(1, 2);
let doc1 = "machine learning is transforming software engineering";
let doc2 = "deep learning and machine learning are related fields";
let doc3 = "quantum physics explores subatomic particles and waves";
let d1: LatticeElement = tok.apply(doc1.as_bytes());
let d2: LatticeElement = tok.apply(doc2.as_bytes());
let d3: LatticeElement = tok.apply(doc3.as_bytes());
println!("doc1 vs doc2:  tau={:.3}  jaccard={:.3}", d1.bss_inclusion(&d2), d1.jaccard_similarity(&d2));
println!("doc1 vs doc3:  tau={:.3}  jaccard={:.3}", d1.bss_inclusion(&d3), d1.jaccard_similarity(&d3));
println!("✅ doc1 more similar to doc2 (both about ML)");
}


---
## 10. Catalog LUT & Homogeneous Consensus

For catalog/enumerable data, each value is hashed with 3 seeds (0, 1, 2).
During materialization, a value is recovered only if ≥ 2 of its 3
seeded positions have bits set — providing collision resistance.

```text
value → seed0:(reg,zeros) + seed1:(reg,zeros) + seed2:(reg,zeros)
consensus: ≥ 2 of 3 positions set → value recovered
```

In [ ]:
{
// Create a CatalogLUT with some values
let values: Vec<&[u8]> = vec![b"alice@example.com", b"bob@example.com", b"carol@example.com"];
let lut = CatalogLUT::from_values(values.iter());

// Show the 3 seed positions for a value
let pos = lut.positions_of(b"alice@example.com").unwrap();
println!("alice@example.com has {} seed positions:", pos.len());
for (i, (reg, zeros)) in pos.iter().enumerate() {
    println!("  seed {i}: reg={reg}, zeros={zeros}");
}
println!("LUT contains {} values", lut.len());
}

In [ ]:
{
// Build HLLSet with all 3 seeds for each value
let emails: Vec<&[u8]> = vec![b"a@x.com", b"b@x.com", b"c@x.com", b"d@x.com", b"e@x.com"];
let mut hllset: HLLSet = HLLSet::new();
for &seed in &[0u64, 1, 2] {
    for &e in &emails {
        let hash = murmur3_hash_seeded(e, seed);
        hllset.add_hash(hash);
    }
}
println!("HLLSet popcount: {}", hllset.popcount());

// Materialize with homogeneous consensus
let lut = CatalogLUT::from_values(emails.iter());
let result = materialize_homogeneous_consensus(&hllset, &lut);
println!("Strategy:   {}", result.strategy);
println!("Confidence: {:.2}%", result.confidence * 100.0);
println!("Recovered:  {:?}", result.flat_strings());
}


In [ ]:
{
// Only seed 0 bits set → should NOT pass 2-of-3 consensus
let mut weak_hllset: HLLSet = HLLSet::new();
let hash = murmur3_hash_seeded(b"alice", 0);
weak_hllset.add_hash(hash);
// Seeds 1 and 2 NOT added — this is a weak signal

let lut = CatalogLUT::from_values(&[b"alice"]);
let result = materialize_homogeneous_consensus(&weak_hllset, &lut);
println!("Confidence: {:.2}%", result.confidence * 100.0);
println!("Tokens recovered: {}", result.flat_strings().len());

if result.flat_strings().is_empty() {
    println!("✅ Correctly REJECTED — only 1 of 3 seeds matched (need ≥ 2)");
} else {
    println!("⚠ Unexpected: token recovered with only 1 seed match");
}
}


In [ ]:
{
// Seeds 0 and 1 set → should PASS 2-of-3 consensus
let mut hllset2: HLLSet = HLLSet::new();
for &seed in &[0u64, 1] {
    let hash = murmur3_hash_seeded(b"bob", seed);
    hllset2.add_hash(hash);
}

let lut = CatalogLUT::from_values(&[b"bob"]);
let result = materialize_homogeneous_consensus(&hllset2, &lut);
println!("Confidence: {:.2}%", result.confidence * 100.0);
println!("Recovered: {:?}", result.flat_strings());

assert!(result.flat_strings().contains(&"bob".to_string()),
        "bob should be recovered with 2-of-3 consensus");
}


In [ ]:
{
// Lua: catalog materialization
let mut rt = DslRuntime::new().unwrap();
rt.exec(r#"
    -- Build HLLSet with multi-seed values via Lua
    local elem = hllset.inscribe({"alice", "bob", "carol"})
    -- Materialize using catalog LUT
    local result = hllset.materialize_catalog(elem, {"alice", "bob", "carol"})
    print("confidence: " .. result.confidence)
    print("tokens found: " .. #result.tokens)
    for i = 1, #result.tokens do
        print("  " .. result.tokens[i])
    end
"#).unwrap();
}


In [ ]:
// Compare heterogeneous vs homogeneous materialization
let values: Vec<&[u8]> = vec![b"user1", b"user2", b"user3", b"user4", b"user5"];

// Build HLLSet with 3-seed catalog method
let mut cat_hllset: HLLSet = HLLSet::new();
for &seed in &[0u64, 1, 2] {
    for &v in &values {
        let hash = murmur3_hash_seeded(v, seed);
        cat_hllset.add_hash(hash);
    }
}

// Catalog LUT materialization
let cat_lut = CatalogLUT::from_values(values.iter());
let cat_result = materialize_homogeneous_consensus(&cat_hllset, &cat_lut);

// Token LUT materialization (single-seed, heterogeneous style)
let tok_lut = TokenLUT::from_tokens(values.iter());
let tok_result = materialize_inlut(&cat_hllset, &tok_lut);

println!("CatalogLUT (3-seed consensus):  conf={:.1}%, tokens={}",
         cat_result.confidence * 100.0, cat_result.flat_strings().len());
println!("TokenLUT  (single-seed lookup): conf={:.1}%, tokens={}",
         tok_result.confidence * 100.0, tok_result.flat_strings().len());
println!();
println!("CatalogLUT is more precise: each value must pass 2-of-3 consensus");
println!("TokenLUT may include more candidates due to hash collisions");

---
## 11. MaterializeEngine — Pluggable Backends

The  trait abstracts materialization across backends:
- **InMemoryEngine** — HashMap LUT (ref, ~1μs)
- **DuckDBEngine** — chunked LUT on IPFS
- **FPGASimEngine** — cycle-accurate FPGA model

 dispatches by LUT key.


In [ ]:
{
let mut engine = InMemoryEngine::new("demo");
engine.build(&[b"alpha",b"beta",b"gamma",b"delta",b"epsilon"]);
let h = HLLSet::from_tokens(&["alpha","gamma"]);
let pos = InMemoryEngine::extract_positions(&h);
println!("Positions: {}", pos.len());
let tokens = engine.materialize(&h, &pos).unwrap();
let r: Vec<String> = tokens.iter().map(|t| String::from_utf8_lossy(t).to_string()).collect();
println!("Recovered: {:?}", r);
println!("Engine: {} ({} tokens)", engine.name(), engine.lut_count());
}


In [ ]:
{
let mut ref_e = InMemoryEngine::new("r"); ref_e.build(&[b"a",b"b"]);
let mut fpga = FPGASimEngine::new("f",3); fpga.build(&[b"a",b"b"]);
let h = HLLSet::from_tokens(&["a"]); let p = InMemoryEngine::extract_positions(&h);
let r = ref_e.materialize(&h,&p).unwrap(); let f = fpga.materialize(&h,&p).unwrap();
println!("Bit-exact: {}  Cycles: {}  HW: {}", r==f, fpga.simulated_cycles(p.len()), fpga.is_hardware());
}


In [ ]:
{
let mut a = InMemoryEngine::new("text"); a.build(&[b"hello",b"world"]);
let mut b = FPGASimEngine::new("text_fpga",3); b.build(&[b"hello",b"world"]);
let mut reg = MaterializeRegistry::new();
reg.register("lut:text:v1",Box::new(a)); reg.register("lut:text:fpga",Box::new(b));
for key in reg.list() { let e = reg.get(&key).unwrap(); println!("{} -> {} (hw={})",key,e.name(),e.is_hardware()); }
let h = HLLSet::from_tokens(&["hello"]); let p = InMemoryEngine::extract_positions(&h);
let e = reg.get("lut:text:fpga").unwrap(); let t = e.materialize(&h,&p).unwrap();
println!("FPGA: {:?}", t.iter().map(|x| String::from_utf8_lossy(x)).collect::<Vec<_>>());
}
